# Laboratorio de preparación para la PC2

Este laboratorio verifica cálculos de toda la Unidad 3. No sustituye la demostración escrita: antes de ejecutar cada bloque, predice el resultado y anota qué teorema lo justifica.

## 0. Herramientas y funciones auxiliares

Trabajaremos con aritmética exacta en SymPy siempre que sea posible.

In [ ]:
import sympy as sp
sp.init_printing()

def datos_espectrales(A):
    lam = sp.symbols('lambda')
    p = sp.factor(A.charpoly(lam).as_expr())
    datos = []
    for valor, mult_alg in A.eigenvals().items():
        base = (A - valor * sp.eye(A.rows)).nullspace()
        datos.append((valor, mult_alg, len(base), base))
    return p, datos

def es_diagonalizable_por_multiplicidades(A):
    return sum(len((A-lam*sp.eye(A.rows)).nullspace())
               for lam in A.eigenvals()) == A.rows

## 1. Transformación, núcleo e imagen

Para el simulacro, la transformación tiene matriz

$$
A=\begin{pmatrix}1&1&0\\0&1&1\end{pmatrix}.
$$

Comprobamos bases y rango-nulidad.

In [ ]:
A_T = sp.Matrix([[1, 1, 0], [0, 1, 1]])
nucleo = A_T.nullspace()
imagen = A_T.columnspace()
rango = A_T.rank()
nulidad = len(nucleo)
assert rango + nulidad == A_T.cols
assert rango == 2
nucleo, imagen, (rango, nulidad)

## 2. Adjunto y relaciones ortogonales

En productos internos estándar, la matriz del adjunto es la transpuesta. La igualdad

$$
\ker(T^*)=\operatorname{Im}(T)^\perp
$$

se comprueba comparando dos núcleos.

In [ ]:
A_adj = A_T.T
ker_adj = A_adj.nullspace()
ortogonal_imagen = A_T.T.nullspace()
assert ker_adj == ortogonal_imagen == []

# Caso ponderado del Ejercicio 4
G = sp.Matrix([[2, 1], [1, 3]])
A = sp.Matrix([[1, 2], [0, -1]])
A_estrella = sp.simplify(G.inv() * A.T * G)
x1, x2, y1, y2 = sp.symbols('x1 x2 y1 y2', real=True)
x = sp.Matrix([x1, x2]); y = sp.Matrix([y1, y2])
assert sp.simplify((A*x).dot(G*y) - x.dot(G*A_estrella*y)) == 0
A_estrella

## 3. Multiplicidades y diagonalización

Comparamos una matriz simétrica diagonalizable, una nilpotente defectuosa y una matriz con un bloque de Jordan.

In [ ]:
M1 = sp.Matrix([[2, 1], [1, 2]])
M2 = sp.Matrix([[0, 1, 0], [0, 0, 1], [0, 0, 0]])
M3 = sp.Matrix([[4, 1, 0], [0, 4, 0], [0, 0, 2]])

for M in (M1, M2, M3):
    p, datos = datos_espectrales(M)
    print('Matriz:'); sp.print_latex(M)
    print('p(lambda) =', p)
    print('(valor, m_a, m_g) =', [(d[0], d[1], d[2]) for d in datos])
    print('Diagonalizable:', es_diagonalizable_por_multiplicidades(M), '\n')

assert es_diagonalizable_por_multiplicidades(M1)
assert not es_diagonalizable_por_multiplicidades(M2)
assert not es_diagonalizable_por_multiplicidades(M3)

## 4. Potencias sin diagonalización

El bloque defectuoso puede escribirse como $4I+N$, con $N^2=0$. Por el binomio,

$$
(4I+N)^n=4^nI+n4^{n-1}N.
$$

In [ ]:
n = sp.symbols('n', integer=True, nonnegative=True)
def potencia_defectuosa(k):
    return sp.Matrix([[4**k, k*4**(k-1), 0],
                      [0, 4**k, 0],
                      [0, 0, 2**k]])

for k in range(1, 9):
    assert potencia_defectuosa(k) == M3**k
potencia_defectuosa(5)

## 5. Diagonalización ortogonal y proyectores

Usamos la matriz simétrica de la PC2 2025-I. En vez de aceptar automáticamente una salida, verificaremos

$$
P^TP=I,\qquad AP=PD,\qquad A=\sum_j\lambda_jP_j.
$$

In [ ]:
S = sp.Matrix([[6, 2, 2], [2, 3, 1], [2, 1, 3]])
u8 = sp.Matrix([2, 1, 1]) / sp.sqrt(6)
u2a = sp.Matrix([0, 1, -1]) / sp.sqrt(2)
u2b = sp.Matrix([-1, 1, 1]) / sp.sqrt(3)
P = sp.Matrix.hstack(u8, u2a, u2b)
D = sp.diag(8, 2, 2)
assert sp.simplify(P.T * P) == sp.eye(3)
assert sp.simplify(S * P - P * D) == sp.zeros(3)
assert sp.simplify(P * D * P.T) == S

Proj8 = sp.simplify((S - 2*sp.eye(3)) / 6)
Proj2 = sp.eye(3) - Proj8
assert Proj8**2 == Proj8 and Proj2**2 == Proj2
assert Proj8*Proj2 == sp.zeros(3)
assert S == 8*Proj8 + 2*Proj2
Proj8, Proj2

## 6. Funciones de una matriz simétrica

Los proyectores permiten calcular potencias y raíces sin multiplicaciones repetidas.

In [ ]:
S6 = 8**6 * Proj8 + 2**6 * Proj2
C = 2 * Proj8 + sp.real_root(2, 3) * Proj2
assert S6 == S**6
assert sp.simplify(C**3 - S) == sp.zeros(3)
S6, C

## 7. Forma cuadrática con parámetros

Para

$$
Q=\alpha x_1^2+x_2^2+x_3^2+2\beta x_2x_3,
$$

el espectro se obtiene separando la dirección $e_1$ y diagonalizando un bloque de tamaño dos.

In [ ]:
alpha, beta = sp.symbols('alpha beta', real=True)
M = sp.Matrix([[alpha, 0, 0], [0, 1, beta], [0, beta, 1]])
Qort = sp.Matrix([[1, 0, 0],
                  [0, 1/sp.sqrt(2), 1/sp.sqrt(2)],
                  [0, 1/sp.sqrt(2), -1/sp.sqrt(2)]])
Dparam = sp.simplify(Qort.T * M * Qort)
assert Dparam == sp.diag(alpha, beta + 1, 1 - beta)
Dparam

## 8. Cociente de Rayleigh

La siguiente exploración numérica no prueba el teorema, pero permite ver sus cotas. Los extremos exactos ya están determinados por el espectro.

In [ ]:
import numpy as np
rng = np.random.default_rng(2026)
S_np = np.array(S, dtype=float)
X = rng.normal(size=(5000, 3))
X /= np.linalg.norm(X, axis=1, keepdims=True)
rayleigh = np.einsum('bi,ij,bj->b', X, S_np, X)
assert rayleigh.min() >= 2 - 1e-12
assert rayleigh.max() <= 8 + 1e-12
rayleigh.min(), rayleigh.max()

## 9. Recurrencia y fórmula cerrada

La recurrencia de la PC2 2026-I se representa mediante una matriz compañera.

In [ ]:
B = sp.Matrix([[3, -2], [1, 0]])
P_rec = sp.Matrix([[2, 1], [1, 1]])
D_rec = sp.diag(2, 1)
assert B == P_rec * D_rec * P_rec.inv()

def x_cerrada(k):
    return 3*2**k - 2

sucesion = [1, 4]
for k in range(1, 9):
    sucesion.append(3*sucesion[-1] - 2*sucesion[-2])
assert sucesion == [x_cerrada(k) for k in range(len(sucesion))]
sucesion

## 10. Desafíos para modificar

1. Cambia una entrada de la matriz defectuosa y busca cuándo recupera una base de vectores propios.
2. Sustituye los parámetros de la forma cuadrática por puntos de cada región y de cada frontera; compara el espectro con la clasificación.
3. Construye los proyectores espectrales del Problema 3 del simulacro.
4. Modifica las condiciones iniciales de la recurrencia y deduce la nueva combinación de $1^n$ y $2^n$.
5. Explica por escrito qué verificaciones computacionales constituyen una prueba general y cuáles solo comprueban un ejemplo.